# Fox Persona Fine-Tuning — 6 Models## Google Colab Edition**Target:** DeepSeek-R1 • GLM-4 • Llama 3.1 • Mistral • Qwen 2.5 • Phi-3.5**Method:** Unsloth QLoRA 4-bit (~8GB VRAM/model)**Dataset:** 176 Fox persona pairs (embedded)**Output:** LoRA + merged 16-bit + GGUF Q4_K_M**Time:** ~15-25 min/model on T4---### Quick Start1. **Runtime → Change runtime type → T4 GPU**2. Run all cells in order (Ctrl+F9)3. Models saved to Google Drive → `/content/drive/MyDrive/fox-models/`> **Free tier:** 4-6h → 2-3 models/session. **Colab Pro:** all 6 in one session.

In [ ]:
# @title 1. Install Dependencies (~2 min)!pip install -q unsloth trl datasets accelerate bitsandbytes xformers!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"import torchprint(f"GPU: {torch.cuda.get_device_name(0)}")print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")assert torch.cuda.is_available(), "ENABLE GPU in Runtime → Change runtime type → T4!"

In [ ]:
# @title 2. Mount Google Drivefrom google.colab import drivedrive.mount('/content/drive')!mkdir -p /content/drive/MyDrive/fox-modelsprint("Google Drive mounted")

In [ ]:
# @title 3. Load Dataset (176 pairs, embedded)import json, gzip, base64_data = "..."dataset_jsonl = gzip.decompress(base64.b64decode(_data)).decode()pairs = [json.loads(line) for line in dataset_jsonl.strip().split("\n")]print(f"Loaded {len(pairs)} conversation pairs")with open("/content/fox_dataset.jsonl", "w") as f:    f.write(dataset_jsonl)print("Saved to /content/fox_dataset.jsonl")

In [ ]:
# @title 4. Model ConfigTARGET_MODELS = {    "deepseek": {"base": "unsloth/DeepSeek-R1-Distill-Llama-8B", "chat_template": "deepseek-r1-distill", "max_seq_length": 4096},    "glm":      {"base": "unsloth/glm-4-9b-chat-1m-bnb-4bit",     "chat_template": "chatglm4",           "max_seq_length": 4096},    "llama":    {"base": "unsloth/Meta-Llama-3.1-8B",             "chat_template": "llama-3",            "max_seq_length": 4096},    "mistral":  {"base": "unsloth/mistral-7b-v0.3",               "chat_template": "mistral",            "max_seq_length": 4096},    "qwen":     {"base": "unsloth/Qwen2.5-7B",                    "chat_template": "qwen-2.5",           "max_seq_length": 4096},    "phi":      {"base": "unsloth/Phi-3.5-mini-instruct",         "chat_template": "phi-3",              "max_seq_length": 4096},}LORA_CONFIG = {    "r": 16, "lora_alpha": 16, "lora_dropout": 0, "bias": "none",    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],    "use_gradient_checkpointing": "unsloth", "random_state": 3407, "use_rslora": True,}print("6 models configured | QLoRA 4-bit | LoRA r=16")

In [ ]:
# @title 5. TRAIN ALL 6 MODELSimport jsonfrom pathlib import Pathfrom datasets import Datasetfrom unsloth import FastLanguageModel, is_bfloat16_supportedfrom unsloth.chat_templates import get_chat_template, standardize_sharegptfrom trl import SFTTrainerfrom transformers import TrainingArgumentsdata = [json.loads(line) for line in open("/content/fox_dataset.jsonl")]dataset = Dataset.from_list(data)print(f"Dataset: {len(dataset)} samples\n")OUTPUT = Path("/content/drive/MyDrive/fox-models")results = {}for model_name, cfg in TARGET_MODELS.items():    print(f"\n{'='*60}")    print(f"{model_name.upper()} — {cfg['base']}")    print(f"{'='*60}")    try:        model, tokenizer = FastLanguageModel.from_pretrained(            model_name=cfg["base"], max_seq_length=cfg["max_seq_length"],            dtype=None, load_in_4bit=True,        )        model = FastLanguageModel.get_peft_model(model, **LORA_CONFIG)        tokenizer = get_chat_template(tokenizer, chat_template=cfg["chat_template"])        ds = standardize_sharegpt(dataset)        ds = ds.map(lambda x: {"text": tokenizer.apply_chat_template(            x["messages"], tokenize=False, add_generation_prompt=False)})        model_output = OUTPUT / f"fox-{model_name}-lora"        trainer = SFTTrainer(            model=model, tokenizer=tokenizer, train_dataset=ds,            dataset_text_field="text", max_seq_length=cfg["max_seq_length"],            dataset_num_proc=2, packing=False,            args=TrainingArguments(                per_device_train_batch_size=2, gradient_accumulation_steps=4,                warmup_ratio=0.01, num_train_epochs=3, learning_rate=2e-4,                fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),                logging_steps=10, optim="adamw_8bit", weight_decay=0.01,                lr_scheduler_type="cosine", seed=3407,                output_dir=str(model_output), save_steps=100, save_total_limit=2,                report_to="none",            ),        )        trainer.train()        model.save_pretrained(str(model_output))        tokenizer.save_pretrained(str(model_output))        model.save_pretrained_merged(str(model_output / "merged_16bit"), tokenizer, save_method="merged_16bit")        model.save_pretrained_gguf(str(model_output / "gguf"), tokenizer, quantization_method="q4_k_m")        print(f"  {model_name} DONE -> {model_output}")        results[model_name] = "OK"        del model, tokenizer, trainer; torch.cuda.empty_cache()    except Exception as e:        print(f"  {model_name} FAILED: {e}")        results[model_name] = f"FAIL: {e}"        torch.cuda.empty_cache()print(f"\n{'='*60}")print("TRAINING COMPLETE")for n, s in results.items(): print(f"  {s} {n}")print(f"\nModels: /content/drive/MyDrive/fox-models/")

In [ ]:
# @title 6. Inference TestMODEL = "deepseek"  # @param ["deepseek", "glm", "llama", "mistral", "qwen", "phi"]from unsloth import FastLanguageModelcfg = TARGET_MODELS[MODEL]model_path = f"/content/drive/MyDrive/fox-models/fox-{MODEL}-lora"model, tokenizer = FastLanguageModel.from_pretrained(    model_name=model_path, max_seq_length=cfg["max_seq_length"],    dtype=None, load_in_4bit=True)FastLanguageModel.for_inference(model)for prompt in ["Siapa kamu?", "Apa task kamu?", "Aktifkan fox"]:    msgs = [{"role": "user", "content": prompt}]    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")    outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)    print(f"\nQ: {prompt}")    print(f"A: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")    print("-"*40)

---## Output Structure```fox-models/├── fox-deepseek-lora/    ← LoRA adapter (load with unsloth)│   ├── merged_16bit/     ← Full model (vLLM/Ollama)│   └── gguf/             ← Q4_K_M (llama.cpp)├── fox-glm-lora/├── fox-llama-lora/├── fox-mistral-lora/├── fox-qwen-lora/└── fox-phi-lora/```## Notes- **Free Colab:** 4-6h → 2-3 models/session. Resume next session for remaining.- **Colab Pro:** All 6 models in 1 session (~2-3h).- T4 GPU 16GB sufficient for 7-9B models + QLoRA 4-bit.- Models in Google Drive are persistent.